# Stage-2 test: audio -> motion generation with style control

Analogous to `test_stage1_style.ipynb`, but stage-2 generates motion **from audio**.
Style is no longer read from a reference clip: it is the explicit 8-dim style-scalar
vector passed straight to the frozen stage-1 decoder. So we can synthesise the *same*
audio in different styles by sweeping those scalars (including the pose disp x speed square).

In [ ]:
from pathlib import Path
import os
import glob
import subprocess
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Resolve project root robustly whether launched from repo root or main/.
cwd = Path.cwd()
if (cwd / 'config').exists():
    project_root = cwd
elif (cwd.parent / 'config').exists():
    project_root = cwd.parent
elif Path('/mnt/fastertalk/config').exists():
    project_root = Path('/mnt/fastertalk')
else:
    raise RuntimeError('Could not locate project root containing config/.')

os.chdir(project_root)
print('Project root:', Path.cwd())

from flame_model.FLAME import FLAMEModel
from renderer.renderer import Renderer
from pytorch3d.transforms import matrix_to_euler_angles

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
flame = FLAMEModel(n_shape=300, n_exp=50).to(device)
renderer = Renderer(render_full_head=True).to(device)
print('Renderer and FLAME ready on', device)

In [ ]:
from utils.config import load_flat_config
from models import get_model

cfg = load_flat_config('config/talkinghead-1kh/stage2.yaml')
cfg.batch_size = 1
cfg.device = device.type

# Stage-2 checkpoint to evaluate.
STAGE2_CKPT = '/mnt/fastertalk/logs/stage2/checkpoints/epoch_115.pt'

# Style axis names read from the dataset stats so they stay in sync with what
# was annotated/trained (e.g. 'pose' appears only when it was included).
stats_path = Path(cfg.data_root) / 'annotations' / 'style_disp_stats.npz'
_stats = np.load(stats_path, allow_pickle=True)
_regions = [str(r) for r in _stats['region_names']]
_features = [str(f) for f in _stats['feature_names']] if 'feature_names' in _stats.files else ['disp', 'speed']
STYLE_NAMES = [f'{r}_{f}' for r in _regions for f in _features]
N_STYLE = len(STYLE_NAMES)
name_to_idx = {n: i for i, n in enumerate(STYLE_NAMES)}

print('Device       :', device)
print('Stage2 ckpt  :', STAGE2_CKPT)
print('Stage1 vqvae :', cfg.vqvae_pretrained_path)
print('Audio model  :', cfg.wav2vec2model_path)
print('Style axes   :', STYLE_NAMES)

In [ ]:
# Build the stage-2 model. Its __init__ already loads the FROZEN stage-1
# autoencoder from cfg.vqvae_pretrained_path; we then overlay the trained
# stage-2 weights (audio encoder + transformer + heads) from STAGE2_CKPT.
def _resolve_checkpoint(path_cfg):
    path_cfg = Path(path_cfg)
    if path_cfg.is_file():
        return path_cfg
    if path_cfg.is_dir():
        ckpts = sorted(path_cfg.glob('epoch_*.pt'))
        return ckpts[-1] if ckpts else None
    return None

model = get_model(cfg).to(device)

ckpt_file = _resolve_checkpoint(STAGE2_CKPT)
if ckpt_file is not None:
    ckpt = torch.load(ckpt_file, map_location=device)
    state_dict = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print('Loaded stage-2 checkpoint:', ckpt_file)
    print('Missing keys:', len(missing), '| Unexpected keys:', len(unexpected))
else:
    print('No valid stage-2 checkpoint found - using freshly-initialised weights.')

model.eval()
# Keep the frozen stage-1 sub-model in eval as well (no BN/dropout drift).
model.autoencoder.eval()
print('Model ready.')

In [ ]:
# --- Audio loading (matches training preprocessing) ---
# Stage-2 generates from audio only: no dataset needed, just a folder of wavs.
import librosa
from transformers import Wav2Vec2FeatureExtractor

audio_processor = Wav2Vec2FeatureExtractor.from_pretrained(cfg.wav2vec2model_path)
AUDIO_DIR = Path('/mnt/fastertalk/demo/audio')

def load_audio(wav_path):
    """Load a wav -> normalised waveform tensor [1, N] on device."""
    speech, _ = librosa.load(str(wav_path), sr=16000)
    feats = np.squeeze(audio_processor(speech, sampling_rate=16000).input_values).astype(np.float32)
    return torch.from_numpy(feats).unsqueeze(0).to(device)  # [1, N]

def list_input_wavs(n=None):
    """Return wav paths found in the input folder."""
    paths = sorted(AUDIO_DIR.glob('*.wav'))
    return paths[:n] if n else paths

input_wavs = list_input_wavs()
print('Audio folder:', AUDIO_DIR)
print('Found', len(input_wavs), 'wavs:')
for p in input_wavs:
    print('  ', p.name)


In [ ]:
# --- Rendering helpers (58-dim blendshapes -> FLAME verts -> frames) ---
def get_vertices_from_blendshapes(expr, gpose, jaw, eyelids):
    expr_tensor = expr.to(device)
    gpose_tensor = gpose.to(device)
    jaw_tensor = jaw.to(device)
    _ = eyelids.to(device)
    target_shape_tensor = torch.zeros(expr_tensor.shape[0], 300, device=device)
    I = matrix_to_euler_angles(torch.cat([torch.eye(3, device=device)[None]], dim=0), 'XYZ')
    eye_r = I.clone().squeeze(); eye_l = I.clone().squeeze()
    eyes = torch.cat([eye_r, eye_l], dim=0).expand(expr_tensor.shape[0], -1)
    pose = torch.cat([gpose_tensor, jaw_tensor], dim=-1)
    verts, _ = flame.forward(
        shape_params=target_shape_tensor,
        expression_params=expr_tensor,
        pose_params=pose,
        eye_pose_params=eyes,
    )
    return verts.detach()

def render_frames(blendshapes):
    """[T,58] blendshapes -> (T,H,W,3) uint8 frames."""
    seq = blendshapes
    verts = get_vertices_from_blendshapes(seq[:, :50], seq[:, 50:53], seq[:, 53:56], seq[:, 56:])
    cam = torch.tensor([5, 0, 0], dtype=torch.float32, device=device).unsqueeze(0)
    cam = cam.expand(verts.shape[0], -1)
    frames = renderer.forward(verts, cam)['rendered_img']  # (T,3,H,W) in [0,1]
    return (frames.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)

def mux_audio(video_file, wav_path, out_file):
    """Attach the driving audio to a rendered mp4 (best-effort, needs ffmpeg)."""
    if wav_path is None or not Path(wav_path).exists():
        return video_file
    cmd = ['ffmpeg', '-y', '-i', str(video_file), '-i', str(wav_path),
           '-c:v', 'copy', '-c:a', 'aac', '-shortest', str(out_file)]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        return out_file
    except Exception as e:
        print('  (audio mux skipped:', e, ')')
        return video_file

In [ ]:
def make_style_vector(overrides=None):
    """Build a [1, N_STYLE] z-scored style vector (0 = dataset-average style).

    overrides: dict mapping axis name (e.g. 'pose_disp') -> value in sigma units.
    """
    vec = torch.zeros(1, N_STYLE, device=device)
    if overrides:
        for name, val in overrides.items():
            vec[0, name_to_idx[name]] = float(val)
    return vec

def generate(audio, style_vec=None, use_quantizer=True):
    """Audio [1,N] + style [1,N_STYLE] -> predicted motion [T,58]."""
    if style_vec is None:
        style_vec = make_style_vector()
    model.eval()
    with torch.no_grad():
        if use_quantizer:
            pred = model.predict(audio, style=style_vec)
        else:
            pred = model.predict_no_quantizer(audio, style=style_vec)
    return pred.squeeze(0)  # [T, 58]

In [ ]:
# --- Sanity check: generate from the first input wav with neutral style ---
assert len(input_wavs) > 0, f'No wavs found in {AUDIO_DIR}. Add some .wav files there.'

_wav = input_wavs[0]
_audio = load_audio(_wav)
_pred = generate(_audio, make_style_vector(), use_quantizer=True)
print('Driving audio        :', _wav.name)
print('Audio samples        :', tuple(_audio.shape))
print('Predicted motion shape:', tuple(_pred.shape))
print('Style axes           :', STYLE_NAMES)


In [ ]:
# --- Generate from a wav with neutral style, save a video with audio ---
WAV = input_wavs[0]
print('Driving audio:', WAV)
audio = load_audio(WAV)

pred = generate(audio, make_style_vector(), use_quantizer=True)
print('Predicted motion shape:', tuple(pred.shape))

out_dir = 'demo/video_style'
os.makedirs(out_dir, exist_ok=True)
frames = render_frames(pred)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(frames[0]); ax.axis('off')
def _upd(t):
    im.set_data(frames[min(t, frames.shape[0] - 1)]); return [im]
ani = animation.FuncAnimation(fig, _upd, frames=frames.shape[0], interval=40, blit=True)
video_file = os.path.join(out_dir, f'{WAV.stem}_neutral.mp4')
ani.save(video_file, writer='ffmpeg', fps=25); plt.close(fig)
final = mux_audio(video_file, WAV, os.path.join(out_dir, f'{WAV.stem}_neutral_audio.mp4'))
print('Saved ->', final)


In [ ]:
# --- Per-axis style sweep: same audio, one video per style axis ---
# Each video shows all sweep values side-by-side: [-2s | -1s | 0 | +1s | +2s].
# Pose axes get a companion held open so the swept axis stays on the realistic
# motion manifold (see stage-1 notebook rationale).
SWEEP_VALUES = (-2.0, -1.0, 0.0, 1.0, 2.0)
POSE_COMPANION_SIGMA = 1.5

COMPANION = {}
if 'pose_disp' in name_to_idx and 'pose_speed' in name_to_idx:
    COMPANION[name_to_idx['pose_disp']]  = name_to_idx['pose_speed']
    COMPANION[name_to_idx['pose_speed']] = name_to_idx['pose_disp']

def render_sweep_axis(audio, axis_idx, axis_name, wav_path=None, values=SWEEP_VALUES,
                      companion_idx=None, companion_value=POSE_COMPANION_SIGMA,
                      use_quantizer=True, out_dir='demo/video_style/sweep'):
    os.makedirs(out_dir, exist_ok=True)
    all_frames = []
    for val in values:
        vec = torch.zeros(1, N_STYLE, device=device)
        if companion_idx is not None:
            vec[0, companion_idx] = companion_value
        vec[0, axis_idx] = val
        pred = generate(audio, vec, use_quantizer=use_quantizer)
        all_frames.append(render_frames(pred))

    T = min(f.shape[0] for f in all_frames)
    n_vals = len(values)
    fig, axes = plt.subplots(1, n_vals, figsize=(4 * n_vals, 4.5), gridspec_kw={'wspace': 0.02})
    ims = []
    for ax, fr, val in zip(axes, all_frames, values):
        ax.axis('off'); ax.set_title(f'{val:+.0f}\u03c3', fontsize=12, pad=3)
        ims.append(ax.imshow(fr[0]))
    title = axis_name
    if companion_idx is not None:
        title += f'   ({STYLE_NAMES[companion_idx]} held @ {companion_value:+.1f}\u03c3)'
    fig.suptitle(title, fontsize=13, y=1.01)
    def update(t):
        for im, fr in zip(ims, all_frames):
            im.set_data(fr[min(t, fr.shape[0] - 1)])
        return ims
    ani = animation.FuncAnimation(fig, update, frames=T, interval=40, blit=True)
    out_path = os.path.join(out_dir, f'{WAV.stem}__{axis_name}_sweep.mp4')
    ani.save(out_path, writer='ffmpeg', fps=25); plt.close(fig)
    final = mux_audio(out_path, wav_path, out_path.replace('.mp4', '_audio.mp4'))
    print('  Saved ->', final)

for i, name in enumerate(STYLE_NAMES):
    comp = COMPANION.get(i)
    suffix = f'  (companion {STYLE_NAMES[comp]}={POSE_COMPANION_SIGMA:+.1f}\u03c3)' if comp is not None else ''
    print(f'Sweeping axis {i}: {name}{suffix}')
    render_sweep_axis(audio, axis_idx=i, axis_name=name, wav_path=WAV, companion_idx=comp)


In [ ]:
# --- Pose square sweep: 5x5 grid over the (disp, speed) plane, same audio ---
# Cols = pose_disp (left->right), Rows = pose_speed (top->bottom).
#   top-left     : still            top-right    : slow large sways
#   bottom-left  : jitter           bottom-right : vigorous natural motion
POSE_GRID_DISP  = (-2.0, -1.0, 0.0, 1.0, 2.0)
POSE_GRID_SPEED = (-2.0, -1.0, 0.0, 1.0, 2.0)

def render_pose_square(audio, wav_path=None, disp_values=POSE_GRID_DISP, speed_values=POSE_GRID_SPEED,
                       use_quantizer=True, out_dir='demo/video_style/sweep'):
    if 'pose_disp' not in name_to_idx or 'pose_speed' not in name_to_idx:
        print('Pose axes not present - skipping square pose sweep.'); return
    di, si = name_to_idx['pose_disp'], name_to_idx['pose_speed']
    os.makedirs(out_dir, exist_ok=True)
    rows, cols = list(speed_values), list(disp_values)
    nrows, ncols = len(rows), len(cols)
    grid = [[None] * ncols for _ in range(nrows)]
    for r, sp in enumerate(rows):
        for c, dp in enumerate(cols):
            vec = torch.zeros(1, N_STYLE, device=device)
            vec[0, di] = dp; vec[0, si] = sp
            pred = generate(audio, vec, use_quantizer=use_quantizer)
            grid[r][c] = render_frames(pred)
    T = min(grid[r][c].shape[0] for r in range(nrows) for c in range(ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows),
                              gridspec_kw={'wspace': 0.02, 'hspace': 0.08})
    axes = np.atleast_2d(axes)
    ims = []
    for r in range(nrows):
        for c in range(ncols):
            ax = axes[r][c]; ax.set_xticks([]); ax.set_yticks([])
            ims.append((ax.imshow(grid[r][c][0]), r, c))
            if r == 0: ax.set_title(f'disp {cols[c]:+.0f}\u03c3', fontsize=11)
            if c == 0: ax.set_ylabel(f'speed {rows[r]:+.0f}\u03c3', fontsize=11)
    fig.suptitle('pose square sweep:  disp (x)  \u00d7  speed (y)', fontsize=14, y=1.005)
    def update(t):
        for im, r, c in ims:
            fr = grid[r][c]; im.set_data(fr[min(t, fr.shape[0] - 1)])
        return [im for im, _, _ in ims]
    ani = animation.FuncAnimation(fig, update, frames=T, interval=40, blit=True)
    out_path = os.path.join(out_dir, f'{WAV.stem}__pose_square_sweep.mp4')
    ani.save(out_path, writer='ffmpeg', fps=25); plt.close(fig)
    final = mux_audio(out_path, wav_path, out_path.replace('.mp4', '_audio.mp4'))
    print('  Saved ->', final)

render_pose_square(audio, wav_path=WAV)


In [ ]:
# --- Coupled pose sweep: raise pose_disp AND pose_speed together (single knob) ---
def render_pose_coupled(audio, wav_path=None, values=SWEEP_VALUES, use_quantizer=True,
                        out_dir='demo/video_style/sweep'):
    if 'pose_disp' not in name_to_idx or 'pose_speed' not in name_to_idx:
        print('Pose axes not present - skipping coupled pose sweep.'); return
    di, si = name_to_idx['pose_disp'], name_to_idx['pose_speed']
    os.makedirs(out_dir, exist_ok=True)
    all_frames = []
    for val in values:
        vec = torch.zeros(1, N_STYLE, device=device)
        vec[0, di] = val; vec[0, si] = val
        pred = generate(audio, vec, use_quantizer=use_quantizer)
        all_frames.append(render_frames(pred))
    T = min(f.shape[0] for f in all_frames)
    n_vals = len(values)
    fig, axes = plt.subplots(1, n_vals, figsize=(4 * n_vals, 4.5), gridspec_kw={'wspace': 0.02})
    ims = []
    for ax, fr, val in zip(axes, all_frames, values):
        ax.axis('off'); ax.set_title(f'{val:+.0f}\u03c3', fontsize=12, pad=3)
        ims.append(ax.imshow(fr[0]))
    fig.suptitle('pose_disp + pose_speed (coupled)', fontsize=13, y=1.01)
    def update(t):
        for im, fr in zip(ims, all_frames):
            im.set_data(fr[min(t, fr.shape[0] - 1)])
        return ims
    ani = animation.FuncAnimation(fig, update, frames=T, interval=40, blit=True)
    out_path = os.path.join(out_dir, f'{WAV.stem}__pose_coupled_sweep.mp4')
    ani.save(out_path, writer='ffmpeg', fps=25); plt.close(fig)
    final = mux_audio(out_path, wav_path, out_path.replace('.mp4', '_audio.mp4'))
    print('  Saved ->', final)

render_pose_coupled(audio, wav_path=WAV)
